# LAB 5-6: sieci splotowe CNN dla PlantVillage

Celem notatnika jest zbudowanie i przetestowanie sieci splotowej do klasyfikacji obraz?w li?ci z bazy PlantVillage. U?ywamy tego samego wariantu danych co w zadaniu MLP: obrazy kolorowe oraz pierwsze 5 klas alfabetycznie z katalogu danych.

Zakres wykonania:

- wczytanie obraz?w z co najmniej 5 klas,
- podzia? na zbiory treningowy, walidacyjny i testowy,
- augmentacja danych treningowych,
- model CNN w PyTorch,
- standardowe wizualizacje dok?adno?ciowe,
- macierz pomy?ek i raport klasyfikacji,
- wizualizacja filtr?w splotowych pierwszej warstwy.

## 1. Import bibliotek i konfiguracja

Ustawiamy ziarna losowo?ci, rozmiar obraz?w i parametry eksperymentu. `MAX_IMAGES_PER_CLASS=600` odpowiada poprzedniemu notatnikowi MLP i skraca trening bez zmiany listy klas.

In [ ]:
from pathlib import Path
import random
import time
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Urz?dzenie:", DEVICE)

IMG_SIZE = 64
NUM_CLASSES = 5
MAX_IMAGES_PER_CLASS = 600
BATCH_SIZE = 32
EPOCHS = 10

## 2. Lokalizacja danych

Dane w tym katalogu s? w `archive (2)/plantvillage dataset/color`. Funkcja poni?ej sprawdza kilka typowych ?cie?ek, ?eby notatnik dzia?a? tak?e po zmianie nazwy folderu.

In [ ]:
def find_data_root():
    candidates = [
        Path("archive") / "plantvillage dataset" / "color",
        Path("archive (2)") / "plantvillage dataset" / "color",
        Path("plantvillage dataset") / "color",
        Path("plantvillage-dataset") / "plantvillage dataset" / "color",
        Path("plantvillage-dataset") / "color",
        Path(r"C:\Users\misie\Downloads\LAB5\archive (2)\plantvillage dataset\color"),
    ]
    for path in candidates:
        if path.exists() and any(child.is_dir() for child in path.iterdir()):
            return path
    raise FileNotFoundError("Nie znaleziono katalogu z obrazami PlantVillage.")

DATA_ROOT = find_data_root()
print("DATA_ROOT:", DATA_ROOT.resolve())

## 3. Wczytanie danych i wyb?r tych samych 5 klas co w MLP

`ImageFolder` traktuje ka?dy podfolder jako jedn? klas?. Wybieramy pierwsze 5 klas alfabetycznie, tak samo jak w poprzednim zadaniu MLP. Etykiety klas s? potem przemapowane na zakres `0..4`.

In [ ]:
base_dataset = datasets.ImageFolder(DATA_ROOT)
all_classes = base_dataset.classes

SELECTED_CLASSES = all_classes[:NUM_CLASSES]
selected_old_ids = [base_dataset.class_to_idx[class_name] for class_name in SELECTED_CLASSES]
old_to_new = {old_idx: new_idx for new_idx, old_idx in enumerate(selected_old_ids)}

selected_indices = []
per_class_counter = {old_idx: 0 for old_idx in selected_old_ids}

# Ograniczamy liczb? obraz?w w ka?dej klasie, ale zachowujemy te same klasy co w MLP.
for sample_idx, (_, old_target) in enumerate(base_dataset.samples):
    if old_target in selected_old_ids:
        if MAX_IMAGES_PER_CLASS is None or per_class_counter[old_target] < MAX_IMAGES_PER_CLASS:
            selected_indices.append(sample_idx)
            per_class_counter[old_target] += 1

targets_new = np.array([old_to_new[base_dataset.samples[i][1]] for i in selected_indices])

print("Liczba wszystkich klas w PlantVillage:", len(all_classes))
print("Wybrane klasy:")
for i, class_name in enumerate(SELECTED_CLASSES):
    print(f"{i}: {class_name} ({sum(targets_new == i)} obraz?w)")
print("Liczba obraz?w w podzbiorze:", len(selected_indices))

## 4. Podzia? train/validation/test

Podzia? jest stratygraficzny, czyli zachowuje podobne proporcje klas w ka?dym zbiorze. To jest wa?ne przy klasyfikacji, bo wynik testowy nie powinien zale?e? od przypadkowego braku kt?rej? klasy.

In [ ]:
train_idx, temp_idx, y_train, y_temp = train_test_split(
    selected_indices,
    targets_new,
    test_size=0.30,
    random_state=SEED,
    stratify=targets_new,
)

val_idx, test_idx, y_val, y_test = train_test_split(
    temp_idx,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print(f"Train: {len(train_idx)} | Validation: {len(val_idx)} | Test: {len(test_idx)}")

## 5. Transformacje i augmentacja

Augmentacj? stosujemy tylko dla zbioru treningowego. Walidacja i test musz? pokazywa? realn? jako?? modelu, wi?c u?ywaj? tylko resize, tensorowania i normalizacji.

In [ ]:
class PlantVillageSubset(datasets.ImageFolder):
    def __init__(self, root, old_to_new, transform=None):
        super().__init__(root, transform=transform)
        self.old_to_new = old_to_new

    def __getitem__(self, index):
        image, old_target = super().__getitem__(index)
        return image, self.old_to_new[old_target]


def get_transforms(augment=False):
    train_steps = [transforms.Resize((IMG_SIZE, IMG_SIZE))]
    if augment:
        train_steps += [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(20),
            transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08)),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
        ]
    train_steps += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]

    eval_steps = [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
    return transforms.Compose(train_steps), transforms.Compose(eval_steps)


def make_loaders(batch_size=BATCH_SIZE, augment=True):
    train_transform, eval_transform = get_transforms(augment=augment)

    train_dataset = PlantVillageSubset(DATA_ROOT, old_to_new, transform=train_transform)
    val_dataset = PlantVillageSubset(DATA_ROOT, old_to_new, transform=eval_transform)
    test_dataset = PlantVillageSubset(DATA_ROOT, old_to_new, transform=eval_transform)

    return (
        DataLoader(Subset(train_dataset, train_idx), batch_size=batch_size, shuffle=True, num_workers=0),
        DataLoader(Subset(val_dataset, val_idx), batch_size=batch_size, shuffle=False, num_workers=0),
        DataLoader(Subset(test_dataset, test_idx), batch_size=batch_size, shuffle=False, num_workers=0),
    )

## 6. Podgl?d danych

Najpierw sprawdzamy rozk?ad klas i przyk?adowe obrazy. To pozwala szybko wykry? b??dn? ?cie?k? albo nieoczekiwany dob?r klas.

In [ ]:
counts = pd.Series(targets_new).value_counts().sort_index()
counts.index = SELECTED_CLASSES

plt.figure(figsize=(11, 4))
sns.barplot(x=counts.index, y=counts.values)
plt.xticks(rotation=35, ha="right")
plt.ylabel("Liczba obraz?w")
plt.title("Rozk?ad wybranych klas")
plt.tight_layout()
plt.show()

In [ ]:
preview_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])
preview_dataset = datasets.ImageFolder(DATA_ROOT, transform=preview_transform)

plt.figure(figsize=(12, 7))
for plot_i, dataset_i in enumerate(selected_indices[:15], start=1):
    image, label = preview_dataset[dataset_i]
    plt.subplot(3, 5, plot_i)
    plt.imshow(image.permute(1, 2, 0))
    plt.title(all_classes[label].split("___")[-1], fontsize=8)
    plt.axis("off")
plt.suptitle("Przyk?adowe obrazy z wybranego podzbioru")
plt.tight_layout()
plt.show()

## 7. Wizualizacja augmentacji

Poni?ej wida? kilka losowych wariant?w tego samego obrazu. Model dostaje dzi?ki temu wi?ksz? r??norodno?? przyk?ad?w i zwykle mniej przeucza si? na treningu.

In [ ]:
raw_image_path, _ = base_dataset.samples[selected_indices[0]]
raw_image = Image.open(raw_image_path).convert("RGB")

augmentation_preview = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
])

plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(augmentation_preview(raw_image))
    plt.axis("off")
plt.suptitle("Przyk?adowe augmentacje jednego obrazu")
plt.tight_layout()
plt.show()

## 8. Model CNN

CNN nie sp?aszcza obrazu od razu jak MLP. Warstwy splotowe ucz? si? lokalnych cech, takich jak kraw?dzie, plamy, przebarwienia i tekstury li?ci. Pooling zmniejsza mapy cech, a klasyfikator na ko?cu zamienia cechy na prawdopodobie?stwa klas.

In [ ]:
class LeafCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * (IMG_SIZE // 8) * (IMG_SIZE // 8), 128),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = LeafCNN().to(DEVICE)
print(model)

## 9. Funkcje treningu i ewaluacji

`run_epoch` dzia?a w dw?ch trybach. Je?eli dostanie optimizer, uczy model. Je?eli optimizer jest `None`, tylko liczy strat? i dok?adno??.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    context = torch.enable_grad() if is_training else torch.no_grad()
    with context:
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            if is_training:
                optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            if is_training:
                loss.backward()
                optimizer.step()

            batch_size = x.size(0)
            total_loss += loss.item() * batch_size
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += batch_size

    return total_loss / total, correct / total


def train_model(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=1e-3, augment=True):
    train_loader, val_loader, test_loader = make_loaders(batch_size=batch_size, augment=augment)
    model = LeafCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    history = []
    best_state = None
    best_val_acc = -1.0
    start = time.time()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = deepcopy(model.state_dict())

        print(
            f"Epoka {epoch:02d}: "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    test_loss, test_acc = run_epoch(model, test_loader, criterion)
    elapsed = time.time() - start
    print(f"Najlepsza walidacja: {best_val_acc:.4f}")
    print(f"Test: loss={test_loss:.4f}, acc={test_acc:.4f}")
    print(f"Czas treningu: {elapsed:.1f} s")

    return model, pd.DataFrame(history), test_acc

## 10. Trening CNN z augmentacj?

Wynik testowy liczymy dopiero po zako?czeniu treningu, na najlepszym stanie modelu wed?ug dok?adno?ci walidacyjnej.

In [ ]:
cnn_model, history, test_acc = train_model(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=1e-3, augment=True)
history

## 11. Wizualizacje dok?adno?ciowe

Krzywe `loss` i `accuracy` pokazuj?, czy model si? uczy oraz czy pojawia si? przeuczenie. Du?a r??nica mi?dzy train i validation accuracy sugeruje overfitting.

In [ ]:
def plot_history(history_df):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="validation")
    axes[0].set_xlabel("Epoka")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Przebieg funkcji straty")
    axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["train_acc"], marker="o", label="train")
    axes[1].plot(history_df["epoch"], history_df["val_acc"], marker="o", label="validation")
    axes[1].set_xlabel("Epoka")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Przebieg dok?adno?ci")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history)

## 12. Macierz pomy?ek i raport klasyfikacji

Macierz pomy?ek pokazuje, kt?re klasy model rozr??nia dobrze, a kt?re myli ze sob?. Raport podaje precision, recall i F1-score osobno dla ka?dej klasy.

In [ ]:
def predict(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(DEVICE))
            y_pred.extend(logits.argmax(dim=1).cpu().numpy())
            y_true.extend(y.numpy())
    return np.array(y_true), np.array(y_pred)

_, _, test_loader = make_loaders(batch_size=64, augment=False)
y_true, y_pred = predict(cnn_model, test_loader)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=SELECTED_CLASSES,
    yticklabels=SELECTED_CLASSES,
)
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.xlabel("Predykcja")
plt.ylabel("Klasa rzeczywista")
plt.title("Macierz pomy?ek: CNN")
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=SELECTED_CLASSES))

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)
per_class_df = pd.DataFrame({
    "class": SELECTED_CLASSES,
    "accuracy": per_class_acc,
})

plt.figure(figsize=(11, 4))
sns.barplot(data=per_class_df, x="class", y="accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=35, ha="right")
plt.ylabel("Accuracy")
plt.title("Dok?adno?? osobno dla ka?dej klasy")
plt.tight_layout()
plt.show()

per_class_df

## 13. Wizualizacja filtr?w splotowych

Pierwsza warstwa splotowa ma filtry `5x5x3`. Ka?dy filtr mo?na potraktowa? jak ma?y obraz RGB. Po treningu filtry zwykle reaguj? na proste lokalne wzorce: kontrast, kraw?dzie, plamy koloru i drobne tekstury.

In [ ]:
def show_first_layer_filters(model):
    first_conv = None
    for layer in model.features:
        if isinstance(layer, nn.Conv2d):
            first_conv = layer
            break

    if first_conv is None:
        raise ValueError("Model nie zawiera warstwy Conv2d.")

    weights = first_conv.weight.detach().cpu().clone()
    n_filters = weights.shape[0]
    n_cols = 8
    n_rows = int(np.ceil(n_filters / n_cols))

    plt.figure(figsize=(12, 2 * n_rows))
    for i in range(n_filters):
        filt = weights[i]
        # Przeskalowanie ka?dego filtra do zakresu 0..1 u?atwia wizualizacj?.
        filt = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
        filt = filt.permute(1, 2, 0).numpy()

        plt.subplot(n_rows, n_cols, i + 1)
        plt.imshow(filt)
        plt.title(f"Filtr {i}", fontsize=8)
        plt.axis("off")

    plt.suptitle("Filtry pierwszej warstwy splotowej CNN")
    plt.tight_layout()
    plt.show()

show_first_layer_filters(cnn_model)

## 14. Dodatkowo: mapy aktywacji pierwszej warstwy

Filtry s? parametrami modelu, a mapy aktywacji pokazuj?, gdzie na konkretnym obrazie dany filtr zareagowa? najmocniej. To pomaga zrozumie? dzia?anie sieci splotowej na przyk?adzie.

In [ ]:
def denormalize_image(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image = tensor.cpu() * std + mean
    return image.clamp(0, 1)


def show_activation_maps(model, loader, max_maps=12):
    model.eval()
    x, y = next(iter(loader))
    image = x[0:1].to(DEVICE)
    true_label = y[0].item()

    first_block = nn.Sequential(*list(model.features.children())[:3]).to(DEVICE)
    with torch.no_grad():
        activations = first_block(image).cpu()[0]

    plt.figure(figsize=(12, 5))
    plt.subplot(3, 5, 1)
    plt.imshow(denormalize_image(x[0]).permute(1, 2, 0))
    plt.title(f"Obraz: {SELECTED_CLASSES[true_label].split('___')[-1]}", fontsize=8)
    plt.axis("off")

    for i in range(min(max_maps, activations.shape[0])):
        plt.subplot(3, 5, i + 2)
        plt.imshow(activations[i], cmap="viridis")
        plt.title(f"Mapa {i}", fontsize=8)
        plt.axis("off")

    plt.suptitle("Mapy aktywacji po pierwszej warstwie splotowej")
    plt.tight_layout()
    plt.show()

_, _, test_loader = make_loaders(batch_size=16, augment=False)
show_activation_maps(cnn_model, test_loader)

## 15. Podsumowanie wynik?w

Zapisujemy najwa?niejsze liczby do pliku CSV, ?eby mo?na by?o ?atwo odnie?? CNN do wynik?w z poprzedniego MLP.

In [ ]:
summary = pd.DataFrame([{
    "model": "LeafCNN",
    "classes": NUM_CLASSES,
    "image_size": IMG_SIZE,
    "max_images_per_class": MAX_IMAGES_PER_CLASS,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "augmentation": True,
    "optimizer": "AdamW",
    "learning_rate": 1e-3,
    "test_accuracy": test_acc,
}])

summary.to_csv("wyniki_cnn_plantvillage_lab5_6.csv", index=False)
print("Zapisano: wyniki_cnn_plantvillage_lab5_6.csv")
summary

## Wnioski do sprawozdania

Sie? CNN jest w?a?ciwsza dla obraz?w ni? MLP, poniewa? zachowuje struktur? przestrzenn? danych i uczy si? lokalnych cech przez filtry splotowe. Augmentacja zwi?ksza r??norodno?? danych treningowych bez dok?adania nowych plik?w, co pomaga ograniczy? przeuczenie. Wizualizacja filtr?w pierwszej warstwy pokazuje, jakie podstawowe wzorce model wykrywa na wej?ciu, natomiast macierz pomy?ek pozwala oceni?, kt?re choroby lub stany li?ci s? dla modelu najbardziej podobne.